# Etapa 1 — Comparación cuantitativa de modelos HAI22.04
Sin entrenamiento, sin recalibración y sin modificación de archivos originales. Compara FULL de DT, RF, ET, XGB y persistencia sobre TEST2–TEST4. Las métricas RMSE/MAE se presentan por señal, sin promediarlas entre escalas distintas.

In [1]:
from pathlib import Path
import re, gzip, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path(r"C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles")
OUT = ROOT / 'analisis' / 'etapa_1'
assert ROOT.is_dir(), f'No existe la carpeta: {ROOT}'
OUT.mkdir(parents=True, exist_ok=True)
TESTS = ('test2','test3','test4')
MODELS = ('DT','RF','ET','XGB','PERSIST')
print('Origen:', ROOT, '\nSalida:', OUT)
for sub in ('metricas','deteccion','predicciones'):
    print(sub, len(list((ROOT/sub).rglob('*'))) if (ROOT/sub).exists() else 'NO EXISTE')

Origen: C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles 
Salida: C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles\analisis\etapa_1
metricas 84
deteccion 210
predicciones 189


## 1. Inventario dirigido de archivos y esquemas
Se leen encabezados únicamente. Si una tabla de detección o episodios usa otra convención, el notebook no inventará sus campos: conservará sus datos y reportará qué no pudo consolidar.

In [2]:
def files_in(folder):
    p=ROOT/folder
    return sorted(x for x in p.rglob('*') if x.is_file()) if p.exists() else []
def read_csv(p, nrows=None, usecols=None):
    with p.open('rb') as f: magic=f.read(2)
    return pd.read_csv(p, compression='gzip' if magic==b'\x1f\x8b' else 'infer', nrows=nrows, usecols=usecols, low_memory=False)
def norm(s):return re.sub(r'[^a-z0-9]+','_',str(s).lower()).strip('_')
def model_of(s):
    u=str(s).upper()
    if 'PERSIST' in u:return 'PERSIST'
    for m in ('XGBOOST','XGB','EXTRA_TREES','EXTRATREES','ET','RANDOM_FOREST','RANDOMFOREST','RF','DECISION_TREE','DECISIONTREE','DT'):
        if re.search(r'(^|[^A-Z0-9])'+m+r'([^A-Z0-9]|$)',u):return {'XGBOOST':'XGB','EXTRA_TREES':'ET','EXTRATREES':'ET','RANDOM_FOREST':'RF','RANDOMFOREST':'RF','DECISION_TREE':'DT','DECISIONTREE':'DT'}.get(m,m)
    return None
def test_of(s):
    match=re.search(r'test[_ -]?([234])',str(s),re.I)
    return 'test'+match.group(1) if match else None
def is_full(s):return bool(re.search(r'(^|[^a-z0-9])full([^a-z0-9]|$)',str(s),re.I))
rows=[]
for folder in ('metricas','deteccion','predicciones'):
    for p in files_in(folder):
        if not (p.name.lower().endswith(('.csv','.csv.gz','.gz'))):continue
        try:
            h=read_csv(p,nrows=0)
            rows.append(dict(carpeta=folder,archivo=str(p.relative_to(ROOT)),columnas=' | '.join(h.columns),modelo=model_of(p.name),test=test_of(p.name),full=is_full(p.name),error=''))
        except Exception as e:rows.append(dict(carpeta=folder,archivo=str(p.relative_to(ROOT)),columnas='',modelo=model_of(p.name),test=test_of(p.name),full=is_full(p.name),error=str(e)))
inventory=pd.DataFrame(rows)
inventory.to_csv(OUT/'01_inventario_dirigido.csv',index=False,encoding='utf-8-sig')
display(inventory[inventory.carpeta.isin(['metricas','deteccion'])].head(30))
print('Archivos:',len(inventory),'Errores encabezado:',int(inventory.error.ne('').sum()))

,carpeta,archivo,columnas,modelo,test,full,error
0,metricas,metricas\comparacion_deteccion.csv,metodo | config | archivo | n | n_alerta | fra...,None,None,False,
1,metricas,metricas\deteccion_DT_FULL.csv,metodo | config | archivo | n | n_alerta | fra...,DT,None,True,
2,metricas,metricas\deteccion_DT_SIN_P1.csv,metodo | config | archivo | n | n_alerta | fra...,DT,None,False,
3,metricas,metricas\deteccion_DT_SIN_P2.csv,metodo | config | archivo | n | n_alerta | fra...,DT,None,False,
4,metricas,metricas\deteccion_DT_SIN_P3.csv,metodo | config | archivo | n | n_alerta | fra...,DT,None,False,
5,metricas,metricas\deteccion_DT_SIN_P4.csv,metodo | config | archivo | n | n_alerta | fra...,DT,None,False,
6,metricas,metricas\deteccion_ET_FULL.csv,metodo | config | archivo | n | n_alerta | fra...,ET,None,True,
7,metricas,metricas\deteccion_ET_SIN_P1.csv,metodo | config | archivo | n | n_alerta | fra...,ET,None,False,
8,metricas,metricas\deteccion_ET_SIN_P2.csv,metodo | config | archivo | n | n_alerta | fra...,ET,None,False,
9,metricas,metricas\deteccion_ET_SIN_P3.csv,metodo | config | archivo | n | n_alerta | fra...,ET,None,False,


Archivos: 462 Errores encabezado: 0


## 2. Métricas predictivas por señal y TEST
Se conserva RMSE y MAE en unidades originales. Los promedios de métricas por señal de diferentes escalas no se calculan.

In [3]:
predict=[]; issues=[]
for p in files_in('metricas'):
    if not re.search(r'^prediccion_.*\.csv(?:\.gz)?$',p.name,re.I):continue
    try:
        d=read_csv(p); d.columns=[norm(x) for x in d.columns]
        required={'variable','componente','rmse','mae'}
        if not required.issubset(d.columns):issues.append(f'{p.name}: faltan {required-set(d.columns)}');continue
        d['modelo']=d['metodo'].map(model_of) if 'metodo' in d else model_of(p.name)
        if 'configuracion' in d:
            d=d[d['configuracion'].astype(str).map(is_full)]
        elif not is_full(p.name):continue
        d['test']=d['archivo'].map(test_of) if 'archivo' in d else test_of(p.name)
        d=d[d['modelo'].isin(MODELS)&d['test'].isin(TESTS)].copy()
        d['fuente']=str(p.relative_to(ROOT))
        predict.append(d[['modelo','test','componente','variable','rmse','mae','fuente']])
    except Exception as e:issues.append(f'{p.name}: {e}')
metrics=pd.concat(predict,ignore_index=True) if predict else pd.DataFrame(columns=['modelo','test','componente','variable','rmse','mae','fuente'])
metrics['rmse']=pd.to_numeric(metrics.rmse,errors='coerce');metrics['mae']=pd.to_numeric(metrics.mae,errors='coerce')
metrics.to_csv(OUT/'02_metricas_por_senal.csv',index=False,encoding='utf-8-sig')
print('Filas:',len(metrics),'Modelos:',sorted(metrics.modelo.unique()),'TEST:',sorted(metrics.test.unique()))
print('Incidencias:',issues[:20]);display(metrics.head())

Filas: 360 Modelos: ['DT', 'ET', 'PERSIST', 'RF', 'XGB'] TEST: ['test2', 'test3', 'test4']
Incidencias: []


,modelo,test,componente,variable,rmse,mae,fuente
0,DT,test2,P1,P1_FT01,7.717825,5.726655,metricas\prediccion_DT_FULL.csv
1,DT,test2,P1,P1_FT02,188.412827,79.054329,metricas\prediccion_DT_FULL.csv
2,DT,test2,P1,P1_FT03,19.391403,14.962177,metricas\prediccion_DT_FULL.csv
3,DT,test2,P1,P1_LIT01,31.037125,22.510300,metricas\prediccion_DT_FULL.csv
4,DT,test2,P1,P1_PIT01,0.083799,0.060072,metricas\prediccion_DT_FULL.csv


## 3. Comparaciones predictivas pareadas frente a persistencia
Cada diferencia se calcula sobre la misma señal y TEST. Valores negativos de ΔRMSE o ΔMAE indican mejora respecto a persistencia. Se cuentan señales que mejoran/empeoran sin agregar sus magnitudes heterogéneas.

In [4]:
base=metrics[metrics.modelo=='PERSIST'][['test','componente','variable','rmse','mae']].rename(columns={'rmse':'rmse_persist','mae':'mae_persist'})
paired=metrics[metrics.modelo!='PERSIST'].merge(base,on=['test','componente','variable'],how='inner',validate='many_to_one') if not base.duplicated(['test','componente','variable']).any() else pd.DataFrame()
if len(paired):
    paired['delta_rmse']=paired.rmse-paired.rmse_persist
    paired['delta_mae']=paired.mae-paired.mae_persist
    paired['mejora_rmse_pct']=np.where(paired.rmse_persist>0,100*(paired.rmse_persist-paired.rmse)/paired.rmse_persist,np.nan)
    paired.to_csv(OUT/'03_comparacion_pareada_por_senal.csv',index=False,encoding='utf-8-sig')
    summary=paired.groupby(['modelo','test','componente'],dropna=False).agg(senales=('variable','nunique'),mejor_rmse=('delta_rmse',lambda x:int((x<0).sum())),peor_rmse=('delta_rmse',lambda x:int((x>0).sum())),igual_rmse=('delta_rmse',lambda x:int((x==0).sum())),mejor_mae=('delta_mae',lambda x:int((x<0).sum()))).reset_index()
    summary.to_csv(OUT/'04_balance_predictivo.csv',index=False,encoding='utf-8-sig')
    display(summary)
else: print('No hay pares comparables. Revisar 02_metricas_por_senal.csv y los nombres de columnas.')

,modelo,test,componente,senales,mejor_rmse,peor_rmse,igual_rmse,mejor_mae
0,DT,test2,P1,9,0,9,0,0
1,DT,test2,P2,6,0,6,0,0
2,DT,test2,P3,3,0,3,0,0
3,DT,test2,P4,6,2,4,0,1
4,DT,test3,P1,9,0,9,0,0
5,DT,test3,P2,6,0,6,0,0
6,DT,test3,P3,3,0,3,0,0
7,DT,test3,P4,6,2,4,0,1
8,DT,test4,P1,9,0,9,0,0
9,DT,test4,P2,6,0,6,0,0


## 4. Consolidación de detección y episodios
Se extraen exclusivamente campos cuya semántica se pueda identificar por nombre. Se conservan las tablas originales de detección como evidencia. No se calculan tasas de episodios desde alertas por registro.

In [5]:
def classify(p):
    s=p.name.lower()
    return 'episodios' if 'episod' in s else 'deteccion' if ('detecc' in s or 'alert' in s) else 'otro'
def identify(d,p):
    cols={norm(c):c for c in d.columns}
    m=model_of(p.name);t=test_of(p.name);full=is_full(p.name)
    for key in ('metodo','modelo','algoritmo'):
        if key in cols and len(d):m=model_of(str(d[cols[key]].iloc[0])) or m
    for key in ('archivo','dataset','conjunto','test'):
        if key in cols and len(d):t=test_of(str(d[cols[key]].iloc[0])) or t
    if 'configuracion' in cols and len(d):full=is_full(str(d[cols['configuracion']].iloc[0]))
    return m,t,full
frames=[]; detection_index=[]
for p in files_in('deteccion'):
    if not p.name.lower().endswith(('.csv','.csv.gz')):continue
    try:
        d=read_csv(p);d.columns=[norm(c) for c in d.columns]
        m,t,full=identify(d,p)
        detection_index.append(dict(archivo=str(p.relative_to(ROOT)),tipo=classify(p),modelo=m,test=t,full=full,filas=len(d),columnas=' | '.join(d.columns)))
        if m in MODELS and t in TESTS and full:
            d['modelo_etapa1']=m;d['test_etapa1']=t;d['fuente_etapa1']=str(p.relative_to(ROOT));d['tipo_etapa1']=classify(p)
            frames.append(d)
    except Exception as e:issues.append(f'{p.name}: {e}')
idx=pd.DataFrame(detection_index)
idx.to_csv(OUT/'05_inventario_deteccion.csv',index=False,encoding='utf-8-sig')
if frames:
    combined=pd.concat(frames,ignore_index=True,sort=False)
    combined.to_csv(OUT/'06_deteccion_original_consolidada.csv',index=False,encoding='utf-8-sig')
    print('Columnas combinadas:',list(combined.columns));display(idx.head(25))
else:print('No se identificaron tablas de detección FULL/TEST; revisar 05_inventario_deteccion.csv')

Columnas combinadas: ['archivo', 'indice', 'timestamp', 'score', 'alerta', 'attack', 'modelo_etapa1', 'test_etapa1', 'fuente_etapa1', 'tipo_etapa1']


,archivo,tipo,modelo,test,full,filas,columnas
0,deteccion\DT_FULL_calibration.csv.gz,otro,DT,None,True,18691,archivo | indice | timestamp | score | alerta
1,deteccion\DT_FULL_test2.csv.gz,otro,DT,test2,True,82770,archivo | indice | timestamp | score | alerta ...
2,deteccion\DT_FULL_test3.csv.gz,otro,DT,test3,True,62370,archivo | indice | timestamp | score | alerta ...
3,deteccion\DT_FULL_test4.csv.gz,otro,DT,test4,True,129570,archivo | indice | timestamp | score | alerta ...
4,deteccion\DT_FULL_train2.csv.gz,otro,DT,None,True,201570,archivo | indice | timestamp | score | alerta
5,deteccion\DT_FULL_train3.csv.gz,otro,DT,None,True,125970,archivo | indice | timestamp | score | alerta
6,deteccion\DT_FULL_train4.csv.gz,otro,DT,None,True,86371,archivo | indice | timestamp | score | alerta
7,deteccion\DT_FULL_train5.csv.gz,otro,DT,None,True,237570,archivo | indice | timestamp | score | alerta
8,deteccion\DT_FULL_train6.csv.gz,otro,DT,None,True,259170,archivo | indice | timestamp | score | alerta
9,deteccion\DT_SIN_P1_calibration.csv.gz,otro,DT,None,False,18691,archivo | indice | timestamp | score | alerta


## 5. Resumen cuantitativo de tablas de detección
La extracción automática es conservadora: busca métricas explícitas en columnas (no interpreta filas arbitrarias como alertas). Si el formato no coincide, queda señalado para adaptar el lector a los encabezados reales.

In [6]:
records=[]
metric_patterns={'episodios_detectados':('episodios_detectados','detected_episodes','n_episodes_detected'), 'episodios_totales':('episodios_totales','total_episodes','n_episodes'), 'alertas_totales':('alertas_totales','total_alerts','n_alerts'), 'alertas_fuera_etiqueta':('alertas_fuera_etiqueta','alerts_outside_label','outside_attack_alerts','false_positive_count')}
for p in files_in('deteccion'):
    if not p.name.lower().endswith(('.csv','.csv.gz')):continue
    try:
        d=read_csv(p);d.columns=[norm(c) for c in d.columns];m,t,full=identify(d,p)
        if m not in MODELS or t not in TESTS or not full:continue
        for dest,aliases in metric_patterns.items():
            found=next((c for c in aliases if c in d.columns),None)
            if found and len(d)==1:
                val=pd.to_numeric(d[found].iloc[0],errors='coerce')
                if pd.notna(val):records.append(dict(modelo=m,test=t,metrica=dest,valor=val,fuente=str(p.relative_to(ROOT))))
    except Exception as e:issues.append(f'{p.name}: {e}')
detected=pd.DataFrame(records,columns=['modelo','test','metrica','valor','fuente'])
detected.to_csv(OUT/'07_metricas_deteccion_identificadas.csv',index=False,encoding='utf-8-sig')
print('Métricas explícitas identificadas:',len(detected));display(detected)
if len(detected):
    for met,group in detected.groupby('metrica'):
        pivot=group.pivot_table(index='modelo',columns='test',values='valor',aggfunc='first').reindex(MODELS)
        pivot.to_csv(OUT/f'08_{met}.csv',encoding='utf-8-sig')
        ax=pivot.plot(kind='bar',figsize=(8,4),rot=0,title=met.replace('_',' ').capitalize())
        ax.set_ylabel('Cantidad');ax.set_xlabel('Modelo');ax.grid(axis='y',alpha=.25)
        plt.tight_layout();plt.savefig(OUT/f'08_{met}.png',dpi=180);plt.close()
else:print('No se hallaron resúmenes escalares reconocibles; el formato real requiere adaptar el lector sin inventar conteos.')

Métricas explícitas identificadas: 0


,modelo,test,metrica,valor,fuente


No se hallaron resúmenes escalares reconocibles; el formato real requiere adaptar el lector sin inventar conteos.


## 6. Figuras predictivas por señal y TEST
Se genera una figura por componente y métrica; no se promedian escalas.

In [7]:
for comp in sorted(metrics.componente.dropna().unique()):
    part=metrics[metrics.componente==comp]
    for metric in ('rmse','mae'):
        for test in TESTS:
            d=part[part.test==test]
            if d.empty:continue
            pivot=d.pivot_table(index='variable',columns='modelo',values=metric,aggfunc='first')
            if pivot.empty:continue
            ax=pivot.plot(kind='bar',figsize=(max(8,len(pivot)*.65),5),rot=65)
            ax.set_title(f'{metric.upper()} por señal — {comp} — {test.upper()}')
            ax.set_ylabel(f'{metric.upper()} (unidad original de cada señal)');ax.set_xlabel('Señal')
            ax.grid(axis='y',alpha=.2);plt.tight_layout()
            safe=re.sub('[^a-zA-Z0-9_-]','_',str(comp))
            plt.savefig(OUT/f'09_{metric}_{safe}_{test}.png',dpi=180,bbox_inches='tight');plt.close()
print('Figuras predictivas generadas:',len(list(OUT.glob('09_*.png'))))

Figuras predictivas generadas: 24


## 7. Verificación final y reporte de cobertura
Los archivos no encontrados o campos no interpretables quedan documentados. No se declara completada la comparación de detección si faltan sus métricas.

In [8]:
expected={(m,t) for m in MODELS for t in TESTS}
actual=set(zip(metrics.modelo,metrics.test))
coverage=pd.DataFrame([dict(modelo=m,test=t,metricas_predictivas=(m,t) in actual,metricas_deteccion=bool(len(detected[(detected.modelo==m)&(detected.test==t)]))) for m,t in sorted(expected)])
coverage.to_csv(OUT/'10_cobertura_etapa1.csv',index=False,encoding='utf-8-sig')
(OUT/'11_incidencias.txt').write_text('\n'.join(issues) if issues else 'Sin errores de lectura registrados',encoding='utf-8')
produced=sorted(p.name for p in OUT.iterdir() if p.is_file())
assert '02_metricas_por_senal.csv' in produced and '10_cobertura_etapa1.csv' in produced
print('Archivos generados:',len(produced));print('\n'.join(produced));display(coverage)
print('IMPORTANTE: detección completa solo si hay métricas verificables para todos los modelos y TEST.')

Archivos generados: 33
01_inventario_dirigido.csv
02_metricas_por_senal.csv
03_comparacion_pareada_por_senal.csv
04_balance_predictivo.csv
05_inventario_deteccion.csv
06_deteccion_original_consolidada.csv
07_metricas_deteccion_identificadas.csv
09_mae_P1_test2.png
09_mae_P1_test3.png
09_mae_P1_test4.png
09_mae_P2_test2.png
09_mae_P2_test3.png
09_mae_P2_test4.png
09_mae_P3_test2.png
09_mae_P3_test3.png
09_mae_P3_test4.png
09_mae_P4_test2.png
09_mae_P4_test3.png
09_mae_P4_test4.png
09_rmse_P1_test2.png
09_rmse_P1_test3.png
09_rmse_P1_test4.png
09_rmse_P2_test2.png
09_rmse_P2_test3.png
09_rmse_P2_test4.png
09_rmse_P3_test2.png
09_rmse_P3_test3.png
09_rmse_P3_test4.png
09_rmse_P4_test2.png
09_rmse_P4_test3.png
09_rmse_P4_test4.png
10_cobertura_etapa1.csv
11_incidencias.txt


,modelo,test,metricas_predictivas,metricas_deteccion
0,DT,test2,True,False
1,DT,test3,True,False
2,DT,test4,True,False
3,ET,test2,True,False
4,ET,test3,True,False
5,ET,test4,True,False
6,PERSIST,test2,True,False
7,PERSIST,test3,True,False
8,PERSIST,test4,True,False
9,RF,test2,True,False


IMPORTANTE: detección completa solo si hay métricas verificables para todos los modelos y TEST.
